# Reveil：在 Colab A100 上訓練馬賽克還原 IC-LoRA

這本 notebook 在同一個 A100 執行階段跑完整流程：

1. 檢查環境、掛載 Google Drive
2. 安裝 Reveil 和 Lightricks 官方訓練器 `ltx-trainer`
3. 下載 LTX-2.5 模型
4. 自動標註原始影片裡要打碼的區域，產生成對訓練資料（無碼 / 打碼）
5. 用官方前處理把配對編碼成 latent
6. 訓練 IC-LoRA：每 250 步存檔到 Drive，斷線後從頭重跑整本 notebook 會自動續訓

**開始前**

- 執行階段 → 變更執行階段類型 → **A100 GPU**（建議開高 RAM）。
- 到 [Lightricks/LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) 同意授權條款，建立 Read token，存成 Colab 密鑰 `HF_TOKEN`（左側鑰匙圖示）。
- 訓練用的影片放在 Drive 的 `reveil/sources/FC2/`（可以是 Drive 捷徑，子資料夾也會讀；放別的地方就改設定裡的 `SOURCES`），**只能放無碼影片**：訓練是在無碼畫面上打假馬賽克，讓模型學著還原成原本的畫面。有碼片（包括 AI 去碼的版本）的畫面本身就不是正確答案，拿來訓練沒有用。影片也要夠多（建議 20 部以上、不同演員和場景），配對太少模型只會背下那幾段。
- 區域標註 `reveil/regions/<偵測模型>/<影片檔名>.json` 會自動產生，也可以自己放（格式見 README）。
- 素材只能用合法取得、所有演出者都確認成年的影片。

**`SMOKE_TEST` 預設是 `True`**：用隨機區域、每部影片 1 個視窗、訓練 20 步，先確認整條流程能跑通（`sources/` 裡至少要有一部影片）。跑通後把 `SMOKE_TEST` 改成 `False` 正式訓練，會先自動標註。

In [ ]:
# ---- 設定 ----
SMOKE_TEST = True  # 隨機區域 + 20 步，只檢查流程；正式訓練改成 False
STEPS = 3000  # 正式訓練步數
REBUILD_DATA = False  # True：加了新影片或標註後，增量產生配對並前處理
RUN_NAME = "ic_lora_512_v1"
CAPTION = "A sharp, detailed, high quality video."  # 所有配對共用；改了要刪掉 Drive 上的 precomputed/
WORKERS = 8  # 同時處理幾部影片
MAX_WINDOWS_PER_VIDEO = 100  # 每部影片最多取幾組配對（一組約 4–5 秒），讓資料平均來自不同影片；None = 不限
# 偵測要打碼的部位。NudeNet（AGPL-3.0）分得出生殖器、肛門等 18 類；一般影片肛門不打碼，所以只取生殖器。
# 也可以改用 "lada_nsfw_detection_model_v1.3"（只有一類 nsfw，會連肛門一起標；兩個類別設定都要改成 []）。
DETECTOR = "nudenet_640m"
DETECT_CLASSES = ["FEMALE_GENITALIA_EXPOSED", "MALE_GENITALIA_EXPOSED"]  # 空 = 模型的全部類別
DETECT_EXCLUDE = ["ANUS_EXPOSED"]  # 從生殖器的框裡挖掉，不打碼
DETECT_CONF = 0.25  # 每一格的門檻；放低一點，框才不會斷斷續續
DETECT_TRACK_SCORE = 0.4  # 一整段的分數中位數低於這個就不打碼（例如手被誤判），預覽上畫成紅框
MIN_SAMPLES = 100  # 正式訓練最少要有幾個配對，不到就不訓練

DRIVE_ROOT = "/content/drive/MyDrive/reveil"  # 跨 session 保存的東西都在這裡
# 訓練用的無碼影片資料夾，可以列好幾個；子資料夾和 Drive 捷徑都會讀。有碼片（sources/ 裡的 AV）不能放進來。
SOURCES = [f"{DRIVE_ROOT}/sources/FC2"]
REGIONS = f"{DRIVE_ROOT}/regions/{DETECTOR}"  # 每個偵測模型各一個資料夾：換模型就重新標註
PREVIEWS = f"{DRIVE_ROOT}/previews/{DETECTOR}"

REVEIL_REPO = "https://github.com/7WayneLee/Reveil"
REVEIL_REF = "claude/great-dirac-b6pl12"  # 合併到 main 後改成 "main"
LTX2_REF = "e98f6d927d83c92ef2be6aec76c8440b19306e39"  # 這本 notebook 對照過的 LTX-2 commit

KIND = "smoke" if SMOKE_TEST else "main"
LOCAL = "/content/work"
MODELS = "/content/models/ltx-2.5"
DATA = f"{LOCAL}/data/{KIND}"  # 配對、dataset.jsonl、.precomputed（本機 SSD）
PRECOMPUTED = f"{DATA}/.precomputed"
DRIVE_PAIRS = f"{DRIVE_ROOT}/pairs/{KIND}"
DRIVE_PRECOMPUTED = f"{DRIVE_ROOT}/precomputed/{KIND}"
RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else "")
TRAINER = "/content/LTX-2/packages/ltx-trainer"
SHARED_CONDITION = "shared_condition.pt"

## 1. 環境檢查

In [ ]:
import os
import shlex
import shutil
import signal
import subprocess
import sys
import threading
import time
from collections import deque
from pathlib import Path


def sh(cmd, cwd=None, show=None):
    # Run a command (list or string), streaming its output; stop on failure with its last lines.
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(part) for part in cmd]
    print("$", show or shlex.join(cmd), flush=True)
    # Colab's inline matplotlib backend exists only inside this kernel, and Colab's PYTHONPATH
    # must not leak into the LTX-2 virtualenv.
    env = dict(os.environ, MPLBACKEND="Agg")
    if cmd[0] == "uv":
        env.pop("PYTHONPATH", None)
    # Its own process group: whatever the command starts (worker processes, the trainer under
    # uv) is stopped with it when it ends, fails or the cell is interrupted.
    process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, start_new_session=True)
    last_lines = deque(maxlen=5)

    def pump():  # in a thread: processes left behind may hold the pipe open after the command exits
        for line in process.stdout:
            print(line, end="", flush=True)
            last_lines.append(line.rstrip())

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    try:
        while process.poll() is None:  # short sleeps, so pressing stop interrupts at once
            time.sleep(0.2)
    finally:
        try:  # stop whatever the command left running: SIGTERM, then SIGKILL after 5 s
            os.killpg(process.pid, signal.SIGTERM)
            for _ in range(25):
                time.sleep(0.2)
                os.killpg(process.pid, 0)
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()
        reader.join()
    if process.returncode:
        raise RuntimeError(f"command failed with exit code {process.returncode}:\n" + "\n".join(last_lines))


def sync(src, dst, exclude=()):
    # Copy files that are new or changed (size or mtime) from src to dst; skip symlinks and
    # the top-level names in `exclude`.
    copied = 0
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        if rel == ".":
            dirs[:] = [d for d in dirs if d not in exclude]
            files = [f for f in files if f not in exclude]
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        for name in files:
            source, target = os.path.join(root, name), os.path.join(dst, rel, name)
            if os.path.islink(source):
                continue
            if (not os.path.exists(target) or os.path.getsize(target) != os.path.getsize(source)
                    or os.path.getmtime(source) > os.path.getmtime(target) + 1):
                shutil.copy2(source, target)
                copied += 1
    print(f"sync {src} -> {dst}: {copied} files copied", flush=True)


def readable_videos(folders):
    # The videos under `folders`, each checked to open and read, so a broken Drive mount stops
    # the run here instead of failing video by video in the long steps.
    from reveil.make_pairs import find_videos

    missing = [folder for folder in folders if not os.path.isdir(folder)]
    if missing:
        raise RuntimeError(f"找不到 {missing}：檢查 SOURCES，或在 Drive 上建好捷徑")
    videos = find_videos([Path(folder) for folder in folders])
    unreadable = []
    for video in videos:
        try:
            with open(video, "rb") as f:
                f.read(1024)
        except OSError as error:
            unreadable.append(f"{video.name}（{error.strerror}）")
    if unreadable:
        raise RuntimeError(f"{len(unreadable)}/{len(videos)} 部影片讀不到，例如 {unreadable[0]}。"
                           "Drive 掛載可能壞了：執行階段 → 中斷連線並刪除執行階段，再整本重跑")
    return videos


query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
name, memory_mib, driver = (p.strip() for p in subprocess.run(query, capture_output=True, text=True, check=True).stdout.splitlines()[0].split(","))
print(f"GPU {name}, {int(memory_mib) / 1024:.0f} GiB, driver {driver}")
if int(memory_mib) < 70_000:
    print("⚠️ 不是 80GB 的卡：標註、配對、前處理照常；訓練會自動改用 INT8 量化（比較慢）")
if int(driver.split(".")[0]) < 580:
    print("⚠️ 驅動低於 580：LTX-2 用的 torch（CUDA 13.2）可能用不了 GPU")
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"/content 可用 {free_gb:.0f} GB，RAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.0f} GB，CPU {os.cpu_count()} 核")
if free_gb < 150:
    print("⚠️ 磁碟偏小：模型約 72 GB，另外要放配對和前處理結果")

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)  # 重跑時重新掛載：Drive 上改過的捷徑才看得到
for path in (REGIONS, DRIVE_PAIRS, DRIVE_PRECOMPUTED, RUN_DIR, DATA):
    os.makedirs(path, exist_ok=True)
for path in SOURCES:
    print(("✓ " if os.path.isdir(path) else "⚠️ 找不到 ") + path)

## 2. 安裝 Reveil 和 ltx-trainer（約 5–10 分鐘）

In [ ]:
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")  # 只有 repo 改成私有時才需要
except Exception:
    github_token = None
url = REVEIL_REPO.replace("https://", f"https://{github_token}@") if github_token else REVEIL_REPO
shutil.rmtree("/content/Reveil", ignore_errors=True)
sh(["git", "clone", "-q", "--depth", "1", "--branch", REVEIL_REF, url, "/content/Reveil"],
   show=f"git clone {REVEIL_REPO} ({REVEIL_REF})")
sh("pip install -q av")
sh("pip install -q --no-deps -e /content/Reveil")  # numpy / opencv / pyyaml are already on Colab
if "/content/Reveil" not in sys.path:  # pip -e 只對新開的 Python 生效；這個核心直接從原始碼 import
    sys.path.insert(0, "/content/Reveil")

sh("pip install -q uv")
if not os.path.isdir("/content/LTX-2/.git"):
    sh("git init -q /content/LTX-2")
    sh("git -C /content/LTX-2 remote add origin https://github.com/Lightricks/LTX-2")
sh(["git", "-C", "/content/LTX-2", "fetch", "-q", "--depth", "1", "origin", LTX2_REF])
sh("git -C /content/LTX-2 checkout -q FETCH_HEAD")
sh("uv sync", cwd="/content/LTX-2")
sh(["uv", "run", "python", "-c", "import torch, ltx_trainer; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
   cwd=TRAINER)

## 3. 下載 LTX-2.5（約 72 GB，存在本機，每個 session 重新下載）

In [ ]:
sh("pip install -q -U huggingface_hub hf_xet")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import hf_hub_download

MODEL_FILES = {
    "transformer": "diffusion_models/ltx-2.5-22b-dev-transformer-bf16.safetensors",
    "text_encoder": "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors",
    "video_vae": "vae/ltx-2.5-video-vae-bf16.safetensors",
    "audio_vae": "vae/ltx-2.5-audio-vae-bf16.safetensors",  # 不訓練音訊，只是讓 split pack 的路徑齊全
}
MODEL_PATHS = {key: hf_hub_download("Lightricks/LTX-2.5", name, local_dir=MODELS) for key, name in MODEL_FILES.items()}
MODEL_PATHS

## 4. 訓練資料

Drive 上已經有前處理好的資料時直接還原，不重做；`REBUILD_DATA = True` 會增量處理新加入的影片（做過的影片和 latent 會跳過）。

In [ ]:
if os.path.isfile(f"{DRIVE_PRECOMPUTED}/{SHARED_CONDITION}"):
    sync(DRIVE_PRECOMPUTED, PRECOMPUTED)
    sh(["python", "-m", "reveil.conditions", "restore", PRECOMPUTED])
NEED_DATA = REBUILD_DATA or not os.path.isfile(f"{PRECOMPUTED}/{SHARED_CONDITION}")
print("需要產生 / 更新資料" if NEED_DATA else "沿用 Drive 上的前處理資料")

### 自動標註

正式訓練、需要產生資料時才執行。偵測模型找的是無碼畫面裡露出的部位（預設只取生殖器），用來決定假馬賽克打在哪裡；它不是用來找馬賽克的。第一次會下載偵測模型存到 Drive 的 `reveil/models/`，之後直接沿用（Lada 的模型是 ultralytics 權重，會先用 `yolo` 指令以獨立程式匯出成 ONNX）。開始前會先確認每部影片都讀得到。標註存在 `reveil/regions/<偵測模型>/`，預覽在 `reveil/previews/<偵測模型>/`，已經有標註檔的影片會跳過。

第一次正式訓練時，建議先選取預覽下面那一格，按 Ctrl+F8（執行階段 → 執行先前的儲存格）跑到預覽為止；確認標得準，再按 Ctrl+F10（執行之後的儲存格）繼續。預覽上綠框是會打碼的區域，紅框是偵測到但分數太低、被丟掉的（例如被誤判的手）。如果框太大或太小、漏標或誤標，就調整 `DETECT_CONF`、`DETECT_TRACK_SCORE`、`DETECT_CLASSES`、`DETECT_EXCLUDE` 再重跑：設定改了，所有影片會自動重新標註（已經產生過配對的話，要先刪掉 `reveil/pairs/main/` 和 `reveil/precomputed/main/`）。

In [ ]:
import urllib.request

DETECTOR_URLS = {  # 依序嘗試；.onnx 直接用，.pt 是 ultralytics 權重，要先匯出
    "nudenet_640m": [  # NudeNet 官方 GitHub 的下載現在要登入，先用 Hugging Face 上的副本
        "https://huggingface.co/spaces/Nymbo/Nudity-Censor/resolve/7955acb6e51995555fe4e5f2fcde7437004053a0/640m.onnx",
        "https://huggingface.co/spaces/xxparthparekhxx/NudeNet-FastAPI/resolve/794a185a301917f1a3505ab3b8d55b268ea81f0e/640m.onnx",
        "https://github.com/notAI-tech/NudeNet/releases/download/v3.4-weights/640m.onnx",
    ],
    "lada_nsfw_detection_model_v1.3": [
        "https://huggingface.co/ladaapp/lada/resolve/main/lada_nsfw_detection_model_v1.3.pt",
    ],
}


def is_model_file(path):
    # A model file, not a login or error page that a download saved in its place.
    if not os.path.isfile(path):
        return False
    with open(path, "rb") as f:
        return not f.read(64).lstrip().startswith((b"<", b"{"))


def fetch_model(urls, path):
    for url in urls:
        try:
            urllib.request.urlretrieve(url, path)
        except OSError as error:
            print(f"下載失敗：{url}（{error}）")
            continue
        if is_model_file(path):
            print(f"下載完成（{os.path.getsize(path) / 1e6:.0f} MB）：{url}")
            return
        print(f"下載到的不是模型檔，換下一個來源：{url}")
    raise RuntimeError(f"{DETECTOR} 的模型檔都下載不到：{urls}")


if NEED_DATA and not SMOKE_TEST:
    print(f"{len(readable_videos(SOURCES))} 部影片都讀得到")
    detector_onnx = f"{DRIVE_ROOT}/models/{DETECTOR}.onnx"
    if not is_model_file(detector_onnx):  # 還沒下載過，或之前存到的是錯誤頁
        urls = DETECTOR_URLS[DETECTOR]
        local = f"/content/detector/{DETECTOR}{Path(urls[0]).suffix}"
        os.makedirs(os.path.dirname(local), exist_ok=True)
        fetch_model(urls, local)
        if local.endswith(".pt"):
            sh("pip install -q ultralytics==8.4.4")  # Lada 用的版本；只用來匯出模型，以獨立程式執行
            sh(["yolo", "export", f"model={local}", "format=onnx", "dynamic=True"])
            local = str(Path(local).with_suffix(".onnx"))
        os.makedirs(os.path.dirname(detector_onnx), exist_ok=True)
        shutil.copy(local, detector_onnx)
    # 只留 GPU 版 onnxruntime：兩版同時裝會互相蓋掉檔案
    sh("pip uninstall -y -q onnxruntime onnxruntime-gpu")
    sh("pip install -q onnxruntime-gpu[cuda,cudnn]")
    annotate = ["python", "-m", "reveil.annotate", "--videos", *SOURCES, "--out", REGIONS,
                "--model", detector_onnx, "--conf", DETECT_CONF, "--workers", WORKERS,
                "--preview-dir", PREVIEWS]
    if DETECT_CLASSES:
        annotate += ["--classes", *DETECT_CLASSES]
    if DETECT_EXCLUDE:
        annotate += ["--exclude-classes", *DETECT_EXCLUDE]
    annotate += ["--min-track-score", DETECT_TRACK_SCORE]
    sh(annotate)

In [ ]:
if NEED_DATA and not SMOKE_TEST:
    from IPython.display import Image, display

    previews = sorted(Path(PREVIEWS).glob("*.jpg"))
    print(f"{len(previews)} 張預覽，這裡顯示前 4 張，其餘在 Drive 的 {PREVIEWS}")
    for path in previews[:4]:
        print(path.name)
        display(Image(filename=str(path), width=960))

In [ ]:
if NEED_DATA:
    videos = readable_videos(SOURCES)
    if not videos:
        raise RuntimeError(f"{', '.join(SOURCES)} 裡沒有影片：先把無碼原始影片放進去")
    if not SMOKE_TEST:
        missing = [v.name for v in videos if not os.path.isfile(f"{REGIONS}/{v.stem}.json")]
        if len(missing) == len(videos):
            raise RuntimeError(
                f"{len(videos)} 部影片都沒有標註檔（例如 {REGIONS}/{videos[0].stem}.json）；"
                "請看上面自動標註那格的輸出"
            )
        if missing:
            print(f"⚠️ {len(missing)} 部影片沒有標註，會被略過：{missing[:5]}")
    print(f"{len(videos)} 部影片")
    if os.listdir(DRIVE_PAIRS):
        sync(DRIVE_PAIRS, DATA)  # 已完成的影片會被跳過
    make_pairs = ["python", "-m", "reveil.make_pairs", "--videos", *SOURCES, "--out", DATA,
                  "--workers", WORKERS, "--caption", CAPTION]
    if SMOKE_TEST:
        make_pairs += ["--random-regions", "--max-windows-per-video", "1"]
    else:
        make_pairs += ["--regions", REGIONS]
        if MAX_WINDOWS_PER_VIDEO:
            make_pairs += ["--max-windows-per-video", MAX_WINDOWS_PER_VIDEO]
    sh(make_pairs)
    sync(DATA, DRIVE_PAIRS, exclude=[".precomputed"])

In [ ]:
if NEED_DATA:
    sh(["uv", "run", "python", "scripts/process_dataset.py", f"{DATA}/dataset.jsonl",
        "--resolution-buckets", "512x512x121",
        "--model-path", MODEL_PATHS["transformer"],
        "--text-encoder-path", MODEL_PATHS["text_encoder"],
        "--video-vae-path", MODEL_PATHS["video_vae"],
        "--audio-vae-path", MODEL_PATHS["audio_vae"],
        "--skip-audio", "--output-dir", PRECOMPUTED], cwd=TRAINER)
    # 每個樣本的 caption embedding 都一樣：只留一份，其餘換成連結，再存到 Drive。
    sh(["python", "-m", "reveil.conditions", "share", PRECOMPUTED, "--dataset", f"{DATA}/dataset.jsonl"])
    sync(PRECOMPUTED, DRIVE_PRECOMPUTED, exclude=["conditions"])

## 5. 訓練

checkpoint 每 250 步存到 `RUN_DIR/checkpoints/`。斷線後從頭重跑整本 notebook：資料從 Drive 還原，訓練從最新的 checkpoint 接著跑。AI Pro 沒有背景執行，分頁要保持開著。

In [ ]:
import glob

import yaml

samples = sum(1 for _ in Path(PRECOMPUTED, "latents").rglob("*.pt"))
print(f"{samples} 個訓練樣本")
if not SMOKE_TEST and samples < MIN_SAMPLES:
    raise RuntimeError(
        f"只有 {samples} 個樣本（MIN_SAMPLES = {MIN_SAMPLES}），訓練 {STEPS} 步只會背下這幾段。"
        "多放一些無碼影片，用 REBUILD_DATA = True 重跑；確定要這樣訓練就調低 MIN_SAMPLES。"
    )
with open("/content/Reveil/configs/ic_lora_512.yaml") as f:
    config = yaml.safe_load(f)
config["model"].update(
    model_path=MODEL_PATHS["transformer"],
    text_encoder_path=MODEL_PATHS["text_encoder"],
    video_vae_path=MODEL_PATHS["video_vae"],
    audio_vae_path=MODEL_PATHS["audio_vae"],
)
config["data"]["preprocessed_data_root"] = PRECOMPUTED
config["output_dir"] = RUN_DIR
config["optimization"]["steps"] = 20 if SMOKE_TEST else STEPS
if SMOKE_TEST:
    config["checkpoints"]["interval"] = 10
if int(memory_mib) < 70_000 and not config["acceleration"]["quantization"]:
    # bf16 的 22B 模型（44 GB）放不進 40GB 的卡：底模量化成 INT8（約 22 GB），LoRA 照常訓練
    config["acceleration"]["quantization"] = "int8-quanto"
    print("40GB 的卡：底模用 INT8 量化訓練，每步比 80GB 慢一些")
done = sorted(glob.glob(f"{RUN_DIR}/checkpoints/*step_*.safetensors"))
config["model"]["load_checkpoint"] = f"{RUN_DIR}/checkpoints" if done else None
print(f"接著 {os.path.basename(done[-1])} 訓練" if done else "從頭開始訓練")
with open(f"{LOCAL}/train.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
sh(["uv", "run", "python", "scripts/train.py", f"{LOCAL}/train.yaml"], cwd=TRAINER)

## 結果

LoRA 在 Drive 的 `reveil/runs/<RUN_NAME>/checkpoints/lora_weights_step_XXXXX.safetensors`。

- 用 `SMOKE_TEST = True` 跑通之後，改回 `False` 正式訓練。
- 改了 `CAPTION`：刪掉 Drive 上的 `reveil/precomputed/main/` 和 `reveil/pairs/main/` 再重跑。
- 推論（把 LoRA 套到 distilled 模型上還原影片）是 Reveil 的下一步。